# Train & compare YOLO11 sizes — Multi-UAV Disaster Response

Trains **YOLO11 at several sizes (n / s / m / l)** on the same data and reports a
comparison table, so you can pick the best size/accuracy trade-off. Works for **Model A**
(detect: person, vehicle) or **Model B** (segment: building_damaged, road_blocked, water).

**Fair comparison rule:** only the model size changes — data, image size, epochs and seed
are identical across runs.

**Before you start:** build + zip the dataset locally and upload the zip to Google Drive:
```bash
# Model A:  make build-datasets ARGS="--sources visdrone sard --copy-images"; (cd data/unified && zip -r -0 detect.zip detect)
# Model B:  make build-datasets ARGS="--sources rescuenet floodnet --copy-images --max-image-side 1280"; (cd data/unified && zip -r -0 seg.zip seg)
# then upload detect.zip / seg.zip to MyDrive/uav/
```
Set **Runtime -> Change runtime type -> GPU** first.


## 1. GPU + install


In [ ]:
!nvidia-smi -L

In [ ]:
%pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configure
Defaults compare all four sizes on the **detect** task. For **segment**: set `TASK='segment'`,
`DATASET_ZIP=.../seg.zip`, `IMGSZ=1024`. Mirrors `configs/perception/model_a.yaml` / `model_b.yaml`.


In [ ]:
TASK        = 'detect'                    # or 'segment'
SIZES       = ['n', 's', 'm', 'l']       # YOLO11 sizes to compare (n=nano ... l=large)
DATASET_ZIP = '/content/drive/MyDrive/uav/detect.zip'   # segment: '.../seg.zip'
PROJECT_DIR = '/content/drive/MyDrive/uav/runs'

IMGSZ    = 1280        # detect: 1280 (tiny objects). segment: use 1024
EPOCHS   = 50          # reduced for the size sweep; retrain the winner longer (see end)
PATIENCE = 15
BATCH    = -1          # auto-fit VRAM (bigger sizes auto-reduce the batch)
SEED     = 0

## 4. Unzip dataset + fix the data.yaml path for Colab


In [ ]:
import zipfile, os, glob, yaml
os.makedirs('/content/data', exist_ok=True)
if not glob.glob('/content/data/**/data.yaml', recursive=True):
    print('Extracting', DATASET_ZIP, '...')
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall('/content/data')
yml = glob.glob('/content/data/**/data.yaml', recursive=True)[0]
root = os.path.dirname(yml)
with open(yml) as f: d = yaml.safe_load(f)
d['path'] = root
with open(yml, 'w') as f: yaml.safe_dump(d, f, sort_keys=False)
print('data.yaml ->', yml); print(d)

## 5. Train each size and collect the comparison
Each run is saved under `PROJECT_DIR/<task>_<size>/`. One size failing (e.g. out-of-memory
on a large model) is recorded and the sweep continues.


In [ ]:
import time, pandas as pd
from ultralytics import YOLO

suffix = '-seg' if TASK == 'segment' else ''
rows = []
for size in SIZES:
    name = f'yolo11{size}{suffix}'
    print(f'\n{"="*60}\nTraining {name}\n{"="*60}')
    try:
        model = YOLO(f'{name}.pt')
        n_params = sum(p.numel() for p in model.model.parameters()) / 1e6
        t0 = time.time()
        model.train(data=yml, task=TASK, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ,
                    batch=BATCH, seed=SEED, project=PROJECT_DIR, name=f'{TASK}_{size}',
                    exist_ok=True, plots=True)
        train_min = (time.time() - t0) / 60
        m = model.val()
        box = m.seg if TASK == 'segment' else m.box
        rows.append({'model': name, 'params_M': round(n_params, 2),
                     'mAP50': round(box.map50, 4), 'mAP50_95': round(box.map, 4),
                     'infer_ms': round(m.speed['inference'], 2),
                     'train_min': round(train_min, 1)})
    except Exception as e:
        print('FAILED:', name, '->', repr(e))
        rows.append({'model': name, 'params_M': None, 'mAP50': None,
                     'mAP50_95': None, 'infer_ms': None, 'train_min': None})

## 6. Comparison table (saved to Drive)


In [ ]:
df = pd.DataFrame(rows).sort_values('mAP50_95', ascending=False, na_position='last')
print(df.to_string(index=False))
out = f'{PROJECT_DIR}/compare_{TASK}.csv'
df.to_csv(out, index=False)
print('\nsaved ->', out)
print('best size:', df.iloc[0]['model'])

## 7. Retrain the winner for the final model
The sweep used `EPOCHS=50` for a fair, cheap comparison. Once you know the best size, train
it properly with the cell below (100 epochs). Its `best.pt` is your final model for Phase 3.

Report the whole table + the winner's final mAP back into `docs/BUILD_LOG.md`.


In [ ]:
WINNER = df.iloc[0]['model'].replace('-seg','').replace('yolo11','')  # e.g. 'm'
final = YOLO(f'yolo11{WINNER}{suffix}.pt')
final.train(data=yml, task=TASK, epochs=100, patience=25, imgsz=IMGSZ, batch=BATCH,
            seed=SEED, project=PROJECT_DIR, name=f'{TASK}_final', exist_ok=True, plots=True)
print('final weights:', f'{PROJECT_DIR}/{TASK}_final/weights/best.pt')

---
**Compute note.** 4 sizes x 2 tasks = 8 runs; `l` at large image size is heavy on a free
T4. If you run low on GPU time, start with `SIZES=['n','s','m']`, or run detect and segment
in separate Colab sessions.

**Next:** download the winners' `best.pt` -> repo `weights/model_a.pt` and `weights/model_b.pt`,
then `uv sync --extra perception` locally for Phase 3 (detection cache).
